# COVID-19 Time Series Analysis (SARIMA)

This notebook converts the original script into a step-by-step analysis workflow.
It loads daily COVID-19 case counts for the United States, performs basic preprocessing,
visualizes the series, examines ACF/PACF, fits a SARIMA model, and generates a 30-day forecast.


## 1. Load packages


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.statespace.sarimax import SARIMAX

os.makedirs("outputs/figures", exist_ok=True)


## 2. Load and preprocess the data

- Source: Google COVID-19 Open Data
- Location: United States (`location_key == "US"`)
- Variable: daily new confirmed cases


In [ ]:
url = "https://storage.googleapis.com/covid19-open-data/v3/epidemiology.csv"
df = pd.read_csv(url)

df_us = df[df["location_key"] == "US"][["date", "new_confirmed"]].copy()
df_us = df_us.dropna()
df_us["new_confirmed"] = df_us["new_confirmed"].clip(lower=0)
df_us["date"] = pd.to_datetime(df_us["date"])
df_us = df_us.sort_values("date")

df_us.head()


## 3. Plot the original time series


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(df_us["date"], df_us["new_confirmed"])
plt.title("Daily COVID Cases (US)")
plt.xlabel("Date")
plt.ylabel("New Cases")
plt.savefig("outputs/figures/time_series.png", dpi=300, bbox_inches="tight")
plt.show()


## 4. First-order differencing


In [ ]:
df_us["diff"] = df_us["new_confirmed"].diff()
df_us[["date", "new_confirmed", "diff"]].head()


## 5. ACF and PACF of the differenced series


In [ ]:
plot_acf(df_us["diff"].dropna(), lags=30)
plt.title("Autocorrelation Function (ACF)")
plt.savefig("outputs/figures/acf.png", dpi=300, bbox_inches="tight")
plt.show()

plot_pacf(df_us["diff"].dropna(), lags=30)
plt.title("Partial Autocorrelation Function (PACF)")
plt.savefig("outputs/figures/pacf.png", dpi=300, bbox_inches="tight")
plt.show()


## 6. Fit the SARIMA model

Based on the exploratory analysis, the model used is:

- Non-seasonal: `(1,1,1)`
- Seasonal: `(1,1,1,7)`


In [ ]:
model = SARIMAX(
    df_us["new_confirmed"],
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False
)

result = model.fit()
print(result.summary())


## 7. Generate a 30-day forecast


In [ ]:
forecast = result.get_forecast(steps=30)
mean = forecast.predicted_mean
ci = forecast.conf_int()

future_dates = pd.date_range(
    start=df_us["date"].iloc[-1] + pd.Timedelta(days=1),
    periods=30,
    freq="D"
)

plt.figure(figsize=(10, 4))
plt.plot(df_us["date"], df_us["new_confirmed"], label="Observed")
plt.plot(future_dates, mean, label="Forecast")
plt.fill_between(future_dates, ci.iloc[:, 0], ci.iloc[:, 1], alpha=0.3)
plt.title("30-Day SARIMA Forecast")
plt.xlabel("Date")
plt.ylabel("New Cases")
plt.legend()
plt.savefig("outputs/figures/forecast.png", dpi=300, bbox_inches="tight")
plt.show()


## 8. Key takeaways

- The series is non-stationary and exhibits multiple waves.
- A clear weekly seasonal pattern is visible.
- A SARIMA(1,1,1)(1,1,1,7) model captures both short-term dependence and weekly seasonality.
- The forecast fluctuates around a relatively stable level, with widening uncertainty over time.
